# Crear dataftames CSV

## Location and Time

In [ ]:
# Llegir el CSV conservant els codis com a text.
from pathlib import Path
import pandas as pd

data_dir = Path('dades')
if not data_dir.is_dir():
    data_dir = Path('VI_Project1/dades')

incendis_gencat = pd.read_csv(
    data_dir / 'Incendis_forestals_a_Catalunya._Anys_2011-2024_20261116.csv',
    dtype='string',
)
# ID propi segons la fila original del CSV (no es un ID oficial).
incendis_gencat['idincident'] = range(1, len(incendis_gencat) + 1)
incendis_gencat.head()

### Short dataset study (before cleaning)
Check NaNs and blank strings, exact repeated source rows, dates, code formats, and numeric area values before creating `locationtimeGenCat`. Compare duplicates across all nine original CSV columns, excluding the generated `idincident`. European numbers such as `5.621,80` mean 5621.80 hectares.


In [ ]:
source_columns = incendis_gencat.columns.drop('idincident').tolist()
source_data = incendis_gencat[source_columns]
missing = source_data.isna() | source_data.apply(lambda s: s.str.strip().eq('').fillna(False))
exact_duplicates = source_data.duplicated(keep='first')
parsed_dates = pd.to_datetime(source_data['DATA INCENDI'], format='%d/%m/%Y', errors='coerce')
invalid_municipality = ~source_data['CODI MUNICIPI'].str.fullmatch(r'[0-9]{6}').fillna(False)
invalid_comarca = ~source_data['CODI COMARCA'].str.fullmatch(r'[0-9]{2}').fillna(False)
area_columns = ['HAARBRADES', 'HANOARBRAD', 'HANOFOREST', 'HAFORESTAL']
# Convert for the audit only; preserve the source dataframe.
area_values = source_data[area_columns].apply(
    lambda s: pd.to_numeric(s.str.replace('.', '', regex=False).str.replace(',', '.', regex=False), errors='coerce')
)
invalid_areas = area_values.isna() & ~missing[area_columns]
forest_total_mismatch = (area_values['HAARBRADES'] + area_values['HANOARBRAD'] - area_values['HAFORESTAL']).abs().gt(0.011)
print(f'Source: {len(source_data):,} rows, {len(source_columns)} columns')
print('Missing/blank values by column:')
print(missing.sum().to_string())
print(f'Exact duplicate rows beyond first: {exact_duplicates.sum()}')
print(f'Invalid/missing dates: {parsed_dates.isna().sum()} | Date range: {parsed_dates.min().date()} to {parsed_dates.max().date()}')
print(f'Malformed/missing municipality codes: {invalid_municipality.sum()} | Comarca codes: {invalid_comarca.sum()}')
print(f'Malformed nonmissing area values: {invalid_areas.sum().sum()} | Negative area values: {area_values.lt(0).sum().sum()}')
print(f'Rows with all four areas equal to zero: {area_values.eq(0).fillna(False).all(axis=1).sum()}')
print(f'Forest totals inconsistent by more than 0.011 ha (where components exist): {forest_total_mismatch.sum()}')
print('Rows with missing/blank values:')
print(incendis_gencat.loc[missing.any(axis=1)].to_string(index=False))


### Findings and cleaning decisions
- **7,545 source rows; one missing value:** `HANOARBRAD` for Mollet del Valles on 04/10/2018. Keep this row because its location and date are complete and area is not used in `locationtimeGenCat`. Leave the missing source area unfilled.
- **97 exact duplicates beyond the first:** remove these before joining locations, keeping the first row and its original generated `idincident`. This leaves **7,448 rows**; gaps in IDs are intentional. Without an official incident ID or time of day, identical records could represent separate fires; removing them is the chosen cleaning rule.
- Dates and municipality/comarca code formats are valid. After European number parsing, there are no malformed nonmissing areas or negative areas; available forest totals agree within rounding tolerance. **1,360 rows have all four areas equal to zero**: retain them because zero is not missing and may reflect rounding or a small incident.
- Compare duplicates across the full original record, not only date/location: different fires can occur in the same municipality on the same day. Keep distinct source records even if their projected location/date fields match.
- Preserve the original source dataframe. The final dataframe uses reference municipality coordinates, not exact fire coordinates. Parse dates with `%d/%m/%Y` into pandas datetime; the source has no time of day. Validate the cleaned row count, unique generated IDs, and completeness of final location/date fields.


In [ ]:
# Deduplicate all original columns, excluding the generated ID.
fires = incendis_gencat.drop_duplicates(subset=source_columns, keep='first').copy()
assert len(fires) == len(incendis_gencat) - int(exact_duplicates.sum())
assert not fires.duplicated(subset=source_columns).any()
assert set(incendis_gencat.loc[missing.any(axis=1), 'idincident']).issubset(set(fires['idincident']))
print(f'Removed {len(incendis_gencat) - len(fires)} exact duplicates; kept {len(fires):,} incidents.')


In [ ]:
# Eliminar el digit de control per obtenir el codi municipal de cinc digits.
municipality_ids = fires['CODI MUNICIPI'].str.strip()
assert municipality_ids.str.fullmatch(r'[0-9]{6}').fillna(False).all(), 'Review missing or malformed municipality IDs'
fires['codi_municipi'] = municipality_ids.str[:5]
fires[['CODI MUNICIPI', 'codi_municipi']].head()

In [ ]:
# Preparar els noms i les divisions territorials amb la taula local.
municipalities = pd.read_csv(
    data_dir / 'corrMpisComsAtps.csv', sep=';', skiprows=3, dtype='string',
    usecols=['Codi municipi', 'Nom municipi', 'Codi província', 'Nom província', 'Nom comarca'],
)
municipalities = municipalities.rename(columns={
    'Nom municipi': 'municipi', 'Codi província': 'idprovincia',
    'Nom província': 'provincia', 'Nom comarca': 'comarca',
})
municipalities['codi_municipi'] = municipalities['Codi municipi'].str.strip().str[:5]
municipalities = municipalities[['codi_municipi', 'idprovincia', 'municipi', 'provincia', 'comarca']]

# Llegir les coordenades municipals i convertir les comes decimals.
coords = pd.read_csv(data_dir / 'MUNICIPIOS.csv', sep=';', encoding='cp1252', dtype='string')
coords = coords.loc[coords['COD_PROV'].isin(['08', '17', '25', '43'])].copy()
coords['codi_municipi'] = coords['COD_INE'].str.strip().str[:5]
coords['latitud'] = pd.to_numeric(coords['LATITUD_ETRS89_REGCAN95'].str.replace(',', '.', regex=False), errors='coerce')
coords['longitud'] = pd.to_numeric(coords['LONGITUD_ETRS89_REGCAN95'].str.replace(',', '.', regex=False), errors='coerce')
coords = coords[['codi_municipi', 'latitud', 'longitud']]

In [ ]:
# Unir per codi municipal sense perdre ni duplicar registres.
merged = (
    fires.merge(municipalities, on='codi_municipi', how='left', validate='many_to_one')
    .merge(coords, on='codi_municipi', how='left', validate='many_to_one')
)
assert len(merged) == len(fires), 'The joins changed the number of fire records'
assert merged['idprovincia'].eq(merged['codi_municipi'].str[:2]).fillna(False).all(), 'Review unmatched or conflicting province IDs'

merged['idprovincia'] = merged['idprovincia'].astype('Int64')
merged['idmunicipio'] = merged['codi_municipi'].str[2:5].astype('Int64')
# El CSV nomes conte la data. Convertir-la a datetime 
merged['data_incendi'] = pd.to_datetime(merged['DATA INCENDI'], format='%d/%m/%Y', errors='raise')
locationtimeGenCat = merged[['idincident', 'data_incendi', 'idprovincia', 'idmunicipio', 'municipi', 'provincia', 'latitud', 'longitud', 'comarca']].copy()
locationtimeGenCat.head()

In [ ]:
# Validate the cleaned location/date result.
assert len(locationtimeGenCat) == len(fires), 'The joins changed the cleaned row count'
assert locationtimeGenCat['idincident'].is_unique
assert not locationtimeGenCat.isna().any().any(), 'Review missing location/date fields'
assert pd.api.types.is_datetime64_any_dtype(locationtimeGenCat['data_incendi'])
print(f'Source rows: {len(incendis_gencat):,} | Removed duplicates: {exact_duplicates.sum()} | Location and Time rows: {len(locationtimeGenCat):,}')
print('Missing values in the final dataframe:')
print(locationtimeGenCat.isna().sum())
locationtimeGenCat.info()


In [ ]:
locationtimeGenCat = locationtimeGenCat.rename(columns={
    "idincident": "numeroparte",
    "data_incendi": "data",
    "idmunicipio": "codi_municipi"
})

locationtimeGenCat.to_csv("Clean data/Clean GenCat Data.csv")